# Abdomen CT — MedNeXt — **Faz 2c: Eğitim + Değerlendirme**

nnUNet v2 ile **aynı preprocessed data** kullanılır — yalnızca trainer farklıdır.

| Boyut | Parametre | Önerilen |
|-------|-----------|----------|
| S     | ~5M       | Hızlı deney |
| B     | ~10M      | Denge |
| M     | ~18M      | Yüksek performans |
| L     | ~62M      | SOTA, en ağır |

**Zorunlu dataset (Faz2a output'u):**
- `nnunet-preprocessed` → `nnunet_preprocessed/`, `splits_out/`, `nifti_val/`

**Opsiyonel dataset'ler:**
- `mednext-checkpoint` → önceki session checkpoint
- `abdomen-acikveri` → ham DICOM (yalnızca `nifti_val/` eksikse)

---

| Adım | Süre (T. yakl.) |
|------|------------------|
| MedNeXt-L eğitim 3d_fullres | 10–14 saat |
| Inference + Değerlendirme | 20–30 dk |

---
## 0. Ortam ve GPU

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB
env_name  = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı! Kaggle: Settings→Accelerator→GPU')
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CUDA  : {torch.version.cuda}')

---
## 1. Kütüphane Kurulumu

MedNeXt, nnUNet v2'yi bağımlılık olarak içerir.

In [ ]:
import importlib, shutil, sysconfig

print('MedNeXt + bağımlılıklar kuruluyor...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/MIC-DKFZ/MedNeXt.git',
     'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
importlib.invalidate_caches()

def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_train', 'nnUNetv2_predict']:
    print(f'  {cmd}: {Path(find_nnunet_cmd(cmd)).exists()}')

---
## 2. Konfigürasyon

**MedNeXt ablasyon parametreleri:**
- `MODEL_SIZE`: `S` | `B` | `M` | `L`
- `KERNEL_SIZE`: `3` (hafif) | `5` (daha iyi bağlam)

In [ ]:
import os, json, shutil, time
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── MedNeXt ablasyon parametreleri ───────────────────────────────────────
MODEL_SIZE        = 'L'     # S | B | M | L
KERNEL_SIZE       = 5       # 3 | 5
CONTINUE_TRAINING = False   # True → checkpoint'ten devam

# ─── Sabit parametreler ────────────────────────────────────────────────────
FOLD         = 0
DATASET_ID   = 100
DATASET_NAME = 'Abdomen'
GITHUB_URL   = 'https://github.com/ramazan2020/abdomen1.git'

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

# ─── Kaggle dataset slug'ları ──────────────────────────────────────────────
PRE_DATASET_SLUG  = 'nnunet-preprocessed'  # ZORUNLU — Faz2a output
CKPT_DATASET_SLUG = 'mednext-checkpoint'   # opsiyonel — devam için
RAW_DATASET_SLUG  = 'abdomen-acikveri'     # opsiyonel — val NIfTI yoksa
# ───────────────────────────────────────────────────────────────────────────

MEDNEXT_TRAINER = f'nnUNetTrainerMedNeXt{MODEL_SIZE}_kernel{KERNEL_SIZE}'

if IS_KAGGLE:
    WORK_DIR    = Path('/kaggle/working')
    TMP_DIR     = Path('/kaggle/tmp')
    _pre_base   = Path(f'/kaggle/input/{PRE_DATASET_SLUG}')
    _ckpt_base  = Path(f'/kaggle/input/{CKPT_DATASET_SLUG}')
    _raw_base   = Path(f'/kaggle/input/{RAW_DATASET_SLUG}')

    if not _pre_base.exists():
        raise FileNotFoundError(
            f'Zorunlu dataset bulunamadı: {_pre_base}\n'
            f'Sağ panelden "{PRE_DATASET_SLUG}" (Faz2a output) ekleyin.'
        )

    NNUNET_PREPROCESSED_P = next(
        (p for p in [_pre_base / 'nnunet_preprocessed', _pre_base]
         if (p / f'Dataset{DATASET_ID}_{DATASET_NAME}').exists()),
        _pre_base / 'nnunet_preprocessed'
    )
    SPLIT_DIR = next(
        (p for p in [_pre_base / 'splits_out', _pre_base / 'splits']
         if p.exists() and any(p.glob('*.csv'))),
        _pre_base / 'splits_out'
    )
    NIFTI_VAL_DIR = next(
        (p for p in [_pre_base / 'nifti_val']
         if p.exists() and any(p.glob('*.nii.gz'))),
        _pre_base / 'nifti_val'
    )
    EGITIM_DIR       = _raw_base / 'Egitim Verisi'
    YARISMA_DIR      = _raw_base / 'Test Verisi'
    BILGI_XLSX       = _raw_base / 'Bilgi.xlsx'
    NIFTI_DIR        = TMP_DIR / 'nifti_val_tmp'
    CHECKPOINT_INPUT = _ckpt_base if _ckpt_base.exists() else None
    RESULTS_P        = WORK_DIR / 'mednext_results'

elif IS_COLAB:
    DRIVE_BASE            = Path('/content/drive/MyDrive/Abdomen')
    EGITIM_DIR            = Path('/content/kaggle_data/Egitim Verisi')
    YARISMA_DIR           = Path('/content/kaggle_data/Test Verisi')
    BILGI_XLSX            = Path('/content/kaggle_data/Bilgi.xlsx')
    NIFTI_DIR             = Path('/content/nifti_val_tmp')
    NIFTI_VAL_DIR         = DRIVE_BASE / 'nifti_val'
    SPLIT_DIR             = DRIVE_BASE / 'splits'
    NNUNET_PREPROCESSED_P = Path('/content/nnunet_preprocessed')
    RESULTS_P             = Path('/content/mednext_results')
    CHECKPOINT_INPUT      = None
    WORK_DIR              = Path('/content')

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError: pass
    PROJECT_ROOT          = Path('.').resolve()
    WORK_DIR              = PROJECT_ROOT / 'outputs'
    EGITIM_DIR            = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(PROJECT_ROOT / 'Egitim Verisi')))
    YARISMA_DIR           = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(PROJECT_ROOT / 'Test Verisi')))
    BILGI_XLSX            = Path(os.environ.get('ABDOMEN_BILGI_XLSX', str(PROJECT_ROOT / 'Bilgi.xlsx')))
    NIFTI_DIR             = WORK_DIR / 'nifti_val_tmp'
    NIFTI_VAL_DIR         = WORK_DIR / 'nifti_val'
    SPLIT_DIR             = WORK_DIR / 'splits'
    NNUNET_PREPROCESSED_P = WORK_DIR / 'nnunet_preprocessed'
    RESULTS_P             = WORK_DIR / 'mednext_results'
    CHECKPOINT_INPUT      = None

RESULTS_P.mkdir(parents=True, exist_ok=True)
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

# MedNeXt nnUNet_results env var'ını kullanır
os.environ['nnUNet_raw']          = str(WORK_DIR / 'nnunet_raw_dummy')
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['ABDOMEN_SPLIT_DIR']   = str(SPLIT_DIR)
os.environ['ABDOMEN_TRAIN_DIR']   = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']    = str(YARISMA_DIR)
os.environ['ABDOMEN_BILGI_XLSX']  = str(BILGI_XLSX)

print(f'Ortam               : {env_name}')
print(f'Model               : MedNeXt-{MODEL_SIZE} kernel{KERNEL_SIZE}')
print(f'Trainer             : {MEDNEXT_TRAINER}')
print(f'CONTINUE_TRAINING   : {CONTINUE_TRAINING}')
print(f'nnUNet_preprocessed : {NNUNET_PREPROCESSED_P}  (✓={NNUNET_PREPROCESSED_P.exists()})')
print(f'SPLIT_DIR           : {SPLIT_DIR}  (✓={SPLIT_DIR.exists()})')
print(f'NIFTI_VAL_DIR       : {NIFTI_VAL_DIR}  (✓={NIFTI_VAL_DIR.exists()})')
print(f'RESULTS_P           : {RESULTS_P}')
if IS_KAGGLE:
    print(f'CHECKPOINT_INPUT    : {CHECKPOINT_INPUT}')

_plans = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}' / 'nnUNetPlans.json'
if not _plans.exists():
    raise FileNotFoundError(
        f'nnUNetPlans.json bulunamadı: {_plans}\n'
        f'Faz2a\'yı tamamlayıp "{PRE_DATASET_SLUG}" dataset olarak ekleyin.'
    )
print('nnUNetPlans.json    : ✓')

---
## 3. GitHub Repo + Splits

In [ ]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.splits     import load_fold, raw_case_id
from src.lifting    import BboxLifter
from src.evaluation import f1_at_iou, top5_f1_mean

manifest    = pd.read_csv(SPLIT_DIR / 'manifest.csv')
train_cases = load_fold(FOLD, 'train')
val_cases   = load_fold(FOLD, 'val')
all_cases   = sorted(set(train_cases + val_cases))

print(f'Manifest  : {len(manifest):,} satır')
print(f'Fold {FOLD} : {len(train_cases)} train + {len(val_cases)} val')

def lift_2d_to_3d(manifest_df, case, delta_z=2, iou_th=0.3):
    return BboxLifter(manifest_df, egitim_dir=EGITIM_DIR,
                      yarisma_dir=YARISMA_DIR, delta_z=delta_z, iou_th=iou_th).lift(case)

---
## 4. Checkpoint Geri Yükleme

`CONTINUE_TRAINING = True` ve `mednext-checkpoint` dataset ekli olmalı.

In [ ]:
if CONTINUE_TRAINING and CHECKPOINT_INPUT is not None:
    _trainer_sub = f'{MEDNEXT_TRAINER}__nnUNetPlans__3d_fullres'
    _dst_ds  = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst_tr  = _dst_ds / _trainer_sub

    if not _dst_tr.exists():
        _src_ds = next(
            (p for p in [
                CHECKPOINT_INPUT / f'Dataset{DATASET_ID}_{DATASET_NAME}',
                CHECKPOINT_INPUT / 'mednext_results' / f'Dataset{DATASET_ID}_{DATASET_NAME}',
            ] if p.exists()),
            CHECKPOINT_INPUT
        )
        print(f'Checkpoint yükleniyor: {_src_ds}')
        _dst_ds.mkdir(parents=True, exist_ok=True)
        _src_tr = _src_ds / _trainer_sub
        if _src_tr.exists():
            shutil.copytree(str(_src_tr), str(_dst_tr))
        else:
            shutil.copytree(str(_src_ds), str(_dst_ds), dirs_exist_ok=True)
        print('  ✓')
    else:
        print('Checkpoint zaten mevcut')
elif CONTINUE_TRAINING and CHECKPOINT_INPUT is None:
    print(f'⚠ CONTINUE_TRAINING=True ama "{CKPT_DATASET_SLUG}" dataset eklenmemiş!')
else:
    print(f'Yeni eğitim (CONTINUE_TRAINING=False)')

---
## 5. MedNeXt Eğitimi

nnUNet preprocessed verisi üzerinde `nnUNetv2_train -tr {MEDNEXT_TRAINER}` çalışır.  
**Kaggle 9 saat session sınırı:** Checkpoint otomatik kaydedilir.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU gerekli!'

_args = [find_nnunet_cmd('nnUNetv2_train'),
         str(DATASET_ID), '3d_fullres', str(FOLD),
         '-tr', MEDNEXT_TRAINER, '--npz']
if CONTINUE_TRAINING:
    _args.append('--c')

print(f'=== MedNeXt Eğitim ===')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Model   : MedNeXt-{MODEL_SIZE} kernel{KERNEL_SIZE}')
print(f'Trainer : {MEDNEXT_TRAINER}')
print(f'Dataset : {DATASET_ID}  Fold: {FOLD}  Devam: {CONTINUE_TRAINING}')
print(f'Results : {RESULTS_P}')
print('─' * 60)

_proc = subprocess.Popen(
    _args, env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()

print('─' * 60)
if _proc.returncode == 0:
    print('MedNeXt eğitim tamamlandı ✓')
else:
    print(f'MedNeXt eğitim kodu: {_proc.returncode} (session kesildi / hata)')
    print('Checkpoint mednext_results/ altında — Save Version ile kaydedin')

if IS_COLAB:
    _src = RESULTS_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = Path('/content/drive/MyDrive/Abdomen/mednext_results') / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists():
        _dst.parent.mkdir(parents=True, exist_ok=True)
        if _dst.exists(): shutil.rmtree(str(_dst))
        shutil.copytree(str(_src), str(_dst))
        print(f"Drive'a yedeklendi: {_dst}")

---
## 6. Val NIfTI Hazırlama

In [ ]:
import SimpleITK as sitk
import pydicom
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

VAL_INPUT_DIR = WORK_DIR / 'val_input'
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

_linked = 0
_missing_cases = []

for c in val_cases:
    raw = raw_case_id(c)
    dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
    if dst.exists(): continue
    src = NIFTI_VAL_DIR / f'case_{raw:05d}_0000.nii.gz'
    if src.exists():
        try: os.symlink(src, dst)
        except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
        _linked += 1
    else:
        _missing_cases.append(c)

print(f'Faz2a nifti_val: {_linked} linklendi  |  eksik: {len(_missing_cases)}')

if _missing_cases and EGITIM_DIR.exists():
    print(f'{len(_missing_cases)} vaka DICOM\'dan üretiliyor...')

    def _dicom_to_nifti(case: str) -> str:
        raw = raw_case_id(case)
        out = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        if out.exists(): return 'skip'
        case_dir = EGITIM_DIR / str(raw)
        if not case_dir.exists(): return f'missing:{case}'
        reader = sitk.ImageSeriesReader()
        names  = reader.GetGDCMSeriesFileNames(str(case_dir))
        if not names: return f'no_dcm:{case}'
        size_map = {}
        for n in names:
            try:
                h = pydicom.dcmread(n, stop_before_pixels=True)
                size_map.setdefault((int(h.Rows), int(h.Columns)), []).append(n)
            except Exception: pass
        if len(size_map) > 1:
            names = max(size_map.values(), key=len)
        reader.SetFileNames(names)
        try:
            sitk.WriteImage(reader.Execute(), str(out))
            return 'ok'
        except Exception as e:
            return f'err:{e}'

    sitk.ProcessObject.GlobalWarningDisplayOff()
    with ThreadPoolExecutor(max_workers=4) as ex:
        _res = list(tqdm(ex.map(_dicom_to_nifti, _missing_cases),
                         total=len(_missing_cases), desc='Val DICOM→NIfTI'))
    sitk.ProcessObject.GlobalWarningDisplayOn()

    for c in _missing_cases:
        raw = raw_case_id(c)
        src = NIFTI_DIR / f'case_{raw:05d}_0000.nii.gz'
        dst = VAL_INPUT_DIR / f'case_{raw:05d}_0000.nii.gz'
        if src.exists() and not dst.exists():
            try: os.symlink(src, dst)
            except (OSError, NotImplementedError): shutil.copy2(str(src), str(dst))
elif _missing_cases:
    print(f'⚠ {len(_missing_cases)} NIfTI eksik ve ham DICOM dataset yok')
    print(f'  Sağ panelden "{RAW_DATASET_SLUG}" ekleyin')

_ready = len(list(VAL_INPUT_DIR.glob('*.nii.gz')))
print(f'Val input hazır: {_ready}/{len(val_cases)} dosya')

---
## 7. MedNeXt Inference

In [ ]:
VAL_OUTPUT_DIR = (RESULTS_P
                  / f'Dataset{DATASET_ID}_{DATASET_NAME}'
                  / f'{MEDNEXT_TRAINER}__nnUNetPlans__3d_fullres'
                  / f'fold_{FOLD}'
                  / 'val_predictions')
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Trainer    : {MEDNEXT_TRAINER}')
print(f'Val input  : {len(list(VAL_INPUT_DIR.glob("*.nii.gz")))} dosya')
print('Inference başlatılıyor...')

_proc = subprocess.Popen(
    [find_nnunet_cmd('nnUNetv2_predict'),
     '-i', str(VAL_INPUT_DIR), '-o', str(VAL_OUTPUT_DIR),
     '-d', str(DATASET_ID), '-c', '3d_fullres',
     '-tr', MEDNEXT_TRAINER, '-f', str(FOLD),
     '--save_probabilities'],
    env={**os.environ},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True
)
for line in _proc.stdout:
    print(line, end='', flush=True)
_proc.wait()
print(f'Inference tamamlandı: {len(list(VAL_OUTPUT_DIR.glob("*.nii.gz")))} mask')

---
## 8. Değerlendirme — F1 Skoru

In [ ]:
from scipy import ndimage

def seg_to_pred_df(pred_dir: Path) -> pd.DataFrame:
    rows = []
    for nii_path in sorted(pred_dir.glob('*.nii.gz')):
        try: raw = int(nii_path.stem.split('_')[1])
        except: continue
        mask      = sitk.GetArrayFromImage(sitk.ReadImage(str(nii_path)))
        prob_path = nii_path.with_suffix('').with_suffix('.npz')
        probs     = np.load(str(prob_path))['probabilities'] if prob_path.exists() else None
        for label_id in range(1, len(SUPER_CLASSES)+1):
            binary = (mask == label_id).astype(np.uint8)
            if binary.sum() == 0: continue
            labeled, n_comp = ndimage.label(binary)
            total_vox = float(binary.sum())
            for comp_id in range(1, n_comp+1):
                cm = (labeled == comp_id)
                coords = np.where(cm)
                z1,z2 = int(coords[0].min()), int(coords[0].max())
                y1,y2 = int(coords[1].min()), int(coords[1].max())
                x1,x2 = int(coords[2].min()), int(coords[2].max())
                score = float(probs[label_id][cm].mean()) if probs is not None else float(cm.sum())/total_vox
                for z in range(z1, z2+1):
                    rows.append({'case':raw,'image_id':z,'class':label_id-1,'score':score,
                                 'x1':float(x1),'y1':float(y1),'x2':float(x2),'y2':float(y2)})
    return pd.DataFrame(rows)

pred_df = seg_to_pred_df(VAL_OUTPUT_DIR)
print(f'Prediction: {len(pred_df):,} satır, {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

gt_rows = []
for case in val_cases:
    raw = raw_case_id(case)
    for b in lift_2d_to_3d(manifest, case):
        for z in range(int(b['z1']), int(b['z2'])+1):
            gt_rows.append({'case':raw,'image_id':z,'class':int(b['class']),
                            'x1':float(b['x1']),'y1':float(b['y1']),
                            'x2':float(b['x2']),'y2':float(b['y2'])})
gt_df = pd.DataFrame(gt_rows)

if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'\n=== MedNeXt-{MODEL_SIZE} kernel{KERNEL_SIZE} — Fold {FOLD} ===')
    print(f'Top-5 Mean F1 : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @0.3 : {detail["micro_f1"]:.4f}')
    print()
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30} P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')

---
## 9. Sonuçları Kaydet

In [ ]:
OUTPUT_DIR = WORK_DIR / f'output_mednext_{MODEL_SIZE}_k{KERNEL_SIZE}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not pred_df.empty:
    pred_df.to_csv(OUTPUT_DIR / f'pred_fold{FOLD}.csv', index=False)
    gt_df.to_csv(OUTPUT_DIR   / f'gt_fold{FOLD}.csv',   index=False)
    summary = {
        'model'        : f'MedNeXt-{MODEL_SIZE}_kernel{KERNEL_SIZE}',
        'trainer'      : MEDNEXT_TRAINER,
        'fold'         : FOLD,
        'val_cases'    : len(val_cases),
        'top5_mean_f1' : top5['top5_mean_f1'],
        'macro_f1_03'  : detail['macro_f1'],
        'micro_f1_03'  : detail['micro_f1'],
        'per_threshold': {str(th): float(f1v) for th, f1v in top5['per_threshold']},
        'per_class'    : {str(k): v for k,v in detail['per_class'].items()},
    }
    (OUTPUT_DIR / f'summary_fold{FOLD}.json').write_text(
        json.dumps(summary, indent=2, default=float)
    )

print(f'Çıktılar: {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

if IS_COLAB:
    _dst = Path('/content/drive/MyDrive/Abdomen') / f'output_mednext_{MODEL_SIZE}_k{KERNEL_SIZE}'
    if _dst.exists(): shutil.rmtree(str(_dst))
    shutil.copytree(str(OUTPUT_DIR), str(_dst))
    print(f"Drive'a kopyalandı: {_dst}")

print('\nTamamlandı!')
if IS_KAGGLE:
    print(f'Save Version → mednext_results/ + output_mednext_{MODEL_SIZE}_k{KERNEL_SIZE}/ kaydedilir')